In [1]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [2]:
aoi = {
    "type": "Polygon",
   
    "coordinates": [
        [
            [
              111.66191298444176,
              -6.737450756719085
            ],
            [
              112.16520539429672,
              -6.737450756719085
            ],
            [
              112.16520539429672,
              -7.125663462269856
            ],
            [
              111.66191298444176,
              -7.125663462269856
            ],
            [
              111.66191298444176,
              -6.737450756719085
            ]
        ]
    ]
}
s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-25", "2026-08-25"],
    spatial_extent={
        "west": 111.66191298444176,
        "south": -7.125663462269856,
        "east": 112.16520539429672,
        "north": -6.737450756719085
    },
    # Disesuaikan dengan data yang dibutuhkan
    bands=["CO"],
)

# Agregasi harian agar tidak ada lebih dari satu data per hari
s5p_co_daily = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Agregasi spasial untuk menghasilkan rata-rata time series per AOI
s5p_co_aoi = s5p_co_daily.aggregate_spatial(reducer="mean", geometries=aoi)

# Simpan hasil sebagai CSV
result = s5p_co_aoi.save_result(format="CSV")

# Jalankan job
job = result.create_job(title="s5p_co_timeseries")
job.start_and_wait()

# Download
job.get_results().download_files("output_co")

0:00:00 Job 'j-2608301412064bfd94c5b9123c6f6455': send 'start'


0:00:02 Job 'j-2608301412064bfd94c5b9123c6f6455': queued (progress 0%)


0:00:07 Job 'j-2608301412064bfd94c5b9123c6f6455': queued (progress 0%)


0:00:14 Job 'j-2608301412064bfd94c5b9123c6f6455': queued (progress 0%)


0:00:22 Job 'j-2608301412064bfd94c5b9123c6f6455': queued (progress 0%)


In [3]:
import pandas as pd

df = pd.read_csv("co.csv")

# pastikan kolom tanggal valid
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ambil hanya bulan dan tahun
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

new_df = pd.DataFrame({
    "date": df['date'],
    "CO": df['CO']
})

new_df.to_csv("CO_Timeseries.csv", index=False)

In [4]:
import pandas as pd

df = pd.read_csv("../../data/polutan/CO_Timeseries.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')


In [5]:
import pandas as pd

df = pd.read_csv("../../data/polutan/SO2_Timeseries.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')


In [6]:
import pandas as pd

df = pd.read_csv("../../data/polutan/NO2_Timeseries.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')


In [7]:
import pandas as pd

df_co = pd.read_csv("CO_Timeseries.csv")
df_no2 = pd.read_csv("NO2_Timeseries.csv")
df_so2 = pd.read_csv("SO2_Timeseries.csv")

dataframe_merged = pd.DataFrame({
    "date": df_co['date'],
    "CO": df_co['CO'],
    "NO2": df_no2['NO2'],
    "SO2": df_so2['SO2']
})

dataframe_merged.to_csv("Polutan_Tuban.csv", index=False)